<table width="100%">
  <tr>
    <td align="left" valign="middle" width="100%">
      <img src="../docs/assets/logo_bristol.png" alt="University of Bristol" width="200" />
    </td>
    <td align="right" valign="middle" width="100%">
      <img src="../docs/assets/logo_ufpe.png" alt="UFPE" width="130" />
      &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
      <img src="../docs/assets/logo_kunumi.png" alt="Kunumi" width="150" />
      &nbsp;&nbsp;
    </td>
  </tr>
</table>

---

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_ROOT = Path('/content/latent-ability-ml')
    if not REPO_ROOT.exists():
        subprocess.check_call([
            'git',
            'clone',
            'https://github.com/manuelfjr/latent-ability-ml.git',
            str(REPO_ROOT),
        ])

    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'numpy',
        'pandas',
        'matplotlib',
        'scipy',
        'scikit-learn',
        'seaborn',
        'joblib',
        'tqdm',
        'birt-gd',
    ])

    for candidate in [REPO_ROOT, REPO_ROOT / 'notebooks', REPO_ROOT / 'utils', REPO_ROOT / 'src']:
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

    from nb_utils import set_root
    PROJECT_ROOT = REPO_ROOT
    print(f'Running in Colab with project root: {PROJECT_ROOT}')
else:
    from nb_utils import set_root
    PROJECT_ROOT = set_root(level=2)
    print(f'Running locally with project root: {PROJECT_ROOT}')


**Environment note.** Run the next code cell first. It loads the repository and the local modules needed for this notebook, especially when opening it in Google Colab.


# Answer 2: Reading Supervised Difficulty through Binary IRT and 2PL Curves

This notebook provides one worked solution path for the Section 2 activity.


## Activity Goals

By the end of this notebook, you should be able to:

- explain why supervised examples can have different difficulty levels;
- create a small binary IRT item bank on a real-valued ability scale;
- manipulate difficulty and discrimination values;
- inspect how 2PL ICCs change when item parameters change;
- interpret what different curve shapes mean in practice.


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.handson import binary_irt_probability, plot_binary_iccs


## Task 1

Create your own item bank with three items:

- one easy item with difficulty near `-1`,
- one medium item with difficulty near `0`,
- one hard item with difficulty near `1`.

Give them different discrimination values and plot the ICCs over `theta` in `[-3, 3]`.


In [ ]:
item_bank = pd.DataFrame(
    [
        {'item': 'easy_item', 'difficulty': -1.00, 'discrimination': 0.80},
        {'item': 'medium_item', 'difficulty': 0.00, 'discrimination': 1.20},
        {'item': 'hard_item', 'difficulty': 1.00, 'discrimination': 1.80},
    ]
)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_binary_iccs(item_bank=item_bank, ax=ax, title='Answer item bank: easy, medium, and hard items')
plt.show()

item_bank


## Task 2

Compute the probability of success for three respondents with abilities `-1`, `0`, and `1`, then discuss how the output changes across easy, medium, and hard items.


In [ ]:
abilities = np.array([-1.0, 0.0, 1.0])

probability_table = pd.DataFrame(
    {
        'item': item_bank['item'],
        'difficulty': item_bank['difficulty'],
        'discrimination': item_bank['discrimination'],
        'P(theta=-1)': [binary_irt_probability(-1.0, row.difficulty, row.discrimination) for row in item_bank.itertuples()],
        'P(theta=0)': [binary_irt_probability(0.0, row.difficulty, row.discrimination) for row in item_bank.itertuples()],
        'P(theta=1)': [binary_irt_probability(1.0, row.difficulty, row.discrimination) for row in item_bank.itertuples()],
    }
)

probability_table


## Task 3

Make one item have **negative discrimination** and inspect the curve. In a teaching discussion, how would you interpret that behavior?


In [ ]:
negative_item_bank = item_bank.copy()
negative_item_bank.loc[negative_item_bank['item'] == 'medium_item', 'discrimination'] = -1.20

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_binary_iccs(
    item_bank=negative_item_bank,
    ax=ax,
    title='Negative discrimination example: one curve now runs against the expected ordering',
)
plt.show()

negative_item_bank


## Reflection Questions

1. Which item best separates medium and high ability?
2. Which item is easiest for low-ability respondents?
3. What does a negative discrimination suggest about an item?
4. Why can ICCs be more informative than average accuracy alone?
5. How does this connect back to the supervised idea that some examples are more difficult than others?
